# 03 — Embeddings contextuales congelados

**Proyecto CORDIS / H2020 · bloque de Transformers**

El Transformer se usa aquí **solo como extractor de características**: sus pesos no se
actualizan. Se extrae un vector por documento y encima se entrena un clasificador lineal
clásico. El fine-tuning (donde sí se actualizan los pesos) va en el notebook 04.

Lo que produce este notebook:

| Salida | Dónde |
|---|---|
| `bert_frozen_l2_val.csv`, `bert_frozen_l2_test.csv` | `results/preds/` |
| `bert_frozen_l2_val_scores.npy`, `..._test_scores.npy` | `results/preds/` |
| `bert_frozen_l2_best.json` | `results/` |
| `fig_token_lengths.png`, `fig_pooling_comparison.png`, `fig_per_class_f1_bert_frozen.png` | `figures/` |
| Embeddings `.npy` (~100 MB) | **Google Drive, NO al repositorio** |

**Antes de ejecutar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**.

## 0. Configuración

Ajusta las rutas de la celda siguiente. Todo lo demás se ejecuta de arriba abajo.

In [ ]:
# ============================ CONFIGURACION ==================================
MODEL_NAME = "bert-base-uncased"   # alternativa: "roberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 32                    # cabe en una T4 con fp16
SEED       = 42
LEVEL      = 2                     # nivel de la jerarquia EuroSciVoc

# Rutas (Drive). Sube ahi H2020.zip y split.csv antes de ejecutar.
DRIVE_DIR  = "/content/drive/MyDrive/cordis"
ZIP_PATH   = f"{DRIVE_DIR}/H2020.zip"
SPLIT_PATH = f"{DRIVE_DIR}/split.csv"
EMB_DIR    = f"{DRIVE_DIR}/embeddings"      # los .npy grandes se quedan aqui

# Salidas locales, que luego copias al repositorio
OUT_PREDS  = "/content/out/results/preds"
OUT_RES    = "/content/out/results"
OUT_FIGS   = "/content/out/figures"
# =============================================================================

import os, random, numpy as np
for d in (EMB_DIR, OUT_PREDS, OUT_RES, OUT_FIGS):
    os.makedirs(d, exist_ok=True)

def set_seeds(s=SEED):
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass
set_seeds()
print("semillas fijadas en", SEED)

In [ ]:
!pip install -q transformers
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("!! SIN GPU: Entorno de ejecucion -> Cambiar tipo de entorno -> T4")

## 1. Datos y etiquetas

La etiqueta se construye con la regla que fija el enunciado:

- se toma la **primera** categoría EuroSciVoc del proyecto;
- se sube por la jerarquía hasta su ancestro de **nivel 2**;
- si esa primera categoría no alcanza el nivel 2, el proyecto **se descarta**.

El control del enunciado es que deben conservarse el **90,31 %** de los proyectos. La celda
lo comprueba con un `assert`: si falla, la regla se ha implementado mal.

> Matiz importante: *«la primera categoría para la que pueda extraerse un nivel 2 válido»*
> también podría leerse como «recorrer los códigos hasta encontrar uno que funcione». Esa
> lectura da 31.982 proyectos (90,40 %) y **no** cuadra con el control. La buena es la de
> arriba: mirar solo el primer código.

In [ ]:
import zipfile, ast, pandas as pd

# Descomprimir H2020.zip (solo la primera vez)
if not os.path.exists("/content/data/projects.xlsx"):
    os.makedirs("/content/data", exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content/data")
print(os.listdir("/content/data"))

In [ ]:
proj = pd.read_excel("/content/data/projects.xlsx")
sci  = pd.read_excel("/content/data/SciVocCodes.xlsx")
print("projects:", proj.shape, "| SciVocCodes:", sci.shape)

# code -> camino en la jerarquia, p.ej. 1275 -> ['27','79','483','1275']
s = sci.dropna(subset=["code"]).copy()
s["code"] = s["code"].astype("int64")
PATH = {c: [p for p in str(f).strip("/").split("/") if p]
        for c, f in zip(s["code"], s["full_code"])}

# nombre legible de cada codigo de nivel 1 y 2
NAME = {}
for fp, fc in zip(s["full_path"], s["full_code"]):
    segs  = [x for x in str(fp).strip("/").split("/") if x]
    parts = [x for x in str(fc).strip("/").split("/") if x]
    for i in range(min(len(segs), len(parts))):
        NAME.setdefault((i + 1, parts[i]), segs[i])

def parse_codes(v):
    if pd.isna(v):
        return []
    try:
        return [int(x) for x in ast.literal_eval(str(v))]
    except Exception:
        return []

def ancestor(codes, level):
    '''Ancestro de nivel `level` del PRIMER codigo del proyecto, o None.'''
    if not codes:
        return None
    p = PATH.get(codes[0])
    return p[level - 1] if p and len(p) >= level else None

proj["codes"] = proj["euroSciVocCode"].apply(parse_codes)
proj["label"] = proj["codes"].apply(lambda c: ancestor(c, LEVEL))

df = proj.dropna(subset=["label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)

pct = 100 * len(df) / len(proj)
print(f"proyectos conservados: {len(df)} de {len(proj)}  ({pct:.2f} %)")
print(f"clases de nivel {LEVEL}: {df['label'].nunique()}")
assert abs(pct - 90.31) < 0.02, f"El control falla: {pct:.2f} %, deberia ser 90.31 %"
print("OK: el control del enunciado cuadra.")

## 2. Partición

Se usa **`splits/split.csv`**, la partición común del grupo. Esto no es opcional: si cada
modelo se evalúa sobre filas distintas, las comparaciones del README no valen nada.

In [ ]:
split = pd.read_csv(SPLIT_PATH)
print(split["split"].value_counts().to_string())

df = df.merge(split, on="projectID", how="inner")
print("\ntras cruzar con la particion:", len(df), "proyectos")

tr = df[df["split"] == "train"].reset_index(drop=True)
va = df[df["split"] == "val"].reset_index(drop=True)
te = df[df["split"] == "test"].reset_index(drop=True)
for nm, part in [("train", tr), ("val", va), ("test", te)]:
    print(f"{nm:6s} {len(part):6d}  ({100*len(part)/len(df):4.1f} %)  clases: {part['label'].nunique()}")

# Texto SIN preprocesar: lo exige el enunciado para los embeddings contextuales
texts_tr, y_tr = tr["summary"].astype(str).tolist(), tr["label"].values
texts_va, y_va = va["summary"].astype(str).tolist(), va["label"].values
texts_te, y_te = te["summary"].astype(str).tolist(), te["label"].values

## 3. Longitud en tokens y truncado

Hay que documentar **qué fracción de los resúmenes supera los 512 tokens** y qué se hace
con ellos. Se truncan; esta celda mide cuántos se ven afectados.

In [ ]:
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

all_texts = texts_tr + texts_va + texts_te
lens = np.array([len(ids) for ids in
                 tokenizer(all_texts, add_special_tokens=True, truncation=False)["input_ids"]])

n_over = int((lens > MAX_LENGTH).sum())
pct_over = 100 * n_over / len(lens)
print(f"tokens: media {lens.mean():.0f} | mediana {np.median(lens):.0f} | "
      f"p95 {np.percentile(lens,95):.0f} | p99 {np.percentile(lens,99):.0f} | max {lens.max()}")
print(f"> {MAX_LENGTH} tokens: {n_over} documentos ({pct_over:.3f} %)  -> se truncan")

plt.figure(figsize=(7, 4))
plt.hist(lens, bins=80, color="steelblue", edgecolor="none")
plt.axvline(MAX_LENGTH, color="crimson", ls="--",
            label=f"max_length = {MAX_LENGTH}\n({pct_over:.2f} % truncados)")
plt.xlabel("tokens por resumen"); plt.ylabel("nº de documentos")
plt.title(f"Longitud en tokens ({MODEL_NAME})")
plt.legend(); plt.tight_layout()
plt.savefig(f"{OUT_FIGS}/fig_token_lengths.png", dpi=150)
plt.show()

## 4. Extracción de los embeddings

Dos estrategias de *pooling* en una sola pasada, para poder compararlas después:

- **mean pooling** — media de los estados ocultos de la última capa, **ponderada por la
  máscara de atención** para no promediar el relleno. Es la opción por defecto razonable:
  usa toda la secuencia.
- **CLS** — el estado del token `[CLS]`. En un modelo *sin* fine-tuning este vector está
  entrenado para la tarea de preentrenamiento (NSP), no para la nuestra, así que suele
  rendir algo peor. Compararlos es barato y es justo lo que pide el enunciado.

Los embeddings se guardan en Drive. **No van al repositorio**: son ~100 MB.

In [ ]:
from transformers import AutoModel
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
if device == "cuda":
    model = model.half()          # fp16: el doble de rapido, sin perdida apreciable

@torch.inference_mode()
def embed(texts, desc=""):
    '''Devuelve (mean_pooled, cls) como arrays float32 de (N, hidden).'''
    out_mean, out_cls = [], []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt").to(device)
        h = model(**enc).last_hidden_state              # (B, T, H)
        mask = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
        mean = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        out_mean.append(mean.float().cpu().numpy())
        out_cls.append(h[:, 0].float().cpu().numpy())
        if (i // BATCH_SIZE) % 50 == 0:
            print(f"  {desc} {i:6d}/{len(texts)}", flush=True)
    return np.vstack(out_mean), np.vstack(out_cls)

tag = MODEL_NAME.replace("/", "_")
emb = {}
for nm, texts in [("train", texts_tr), ("val", texts_va), ("test", texts_te)]:
    f_mean = f"{EMB_DIR}/{tag}_{nm}_mean.npy"
    f_cls  = f"{EMB_DIR}/{tag}_{nm}_cls.npy"
    if os.path.exists(f_mean) and os.path.exists(f_cls):
        print(f"{nm}: ya estaban en Drive, se cargan")
        emb[(nm, "mean")] = np.load(f_mean); emb[(nm, "cls")] = np.load(f_cls)
    else:
        print(f"{nm}: extrayendo {len(texts)} documentos...")
        m, c = embed(texts, desc=nm)
        np.save(f_mean, m); np.save(f_cls, c)
        emb[(nm, "mean")] = m; emb[(nm, "cls")] = c
    print(f"   {nm}: {emb[(nm,'mean')].shape}")

## 5. Clasificador lineal sobre los embeddings

Los embeddings son densos y con escalas muy distintas por dimensión, así que se
estandarizan. El `StandardScaler` va **dentro del `Pipeline`**: se ajusta solo con
entrenamiento y se aplica a validación y test, nunca al revés.

La búsqueda recorre `C` y `class_weight`, y se decide por **macro-F1 en validación**.
El test no se toca aquí.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score

GRID_C  = [0.01, 0.1, 1.0, 10.0]
GRID_CW = [None, "balanced"]

rows = []
for pooling in ("mean", "cls"):
    Xtr, Xva = emb[("train", pooling)], emb[("val", pooling)]
    for C in GRID_C:
        for cw in GRID_CW:
            clf = make_pipeline(
                StandardScaler(),
                LinearSVC(C=C, class_weight=cw, random_state=SEED, max_iter=5000),
            ).fit(Xtr, y_tr)
            p = clf.predict(Xva)
            rows.append({"pooling": pooling, "C": C,
                         "class_weight": str(cw),
                         "macro_f1": f1_score(y_va, p, average="macro"),
                         "accuracy": accuracy_score(y_va, p)})
            print(f"{pooling:5s} C={C:<6} cw={str(cw):9s} "
                  f"macroF1={rows[-1]['macro_f1']:.4f}  acc={rows[-1]['accuracy']:.4f}")

search = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
search.to_csv(f"{OUT_RES}/bert_frozen_search_l2.csv", index=False)
best = search.iloc[0]
print("\nMEJOR EN VALIDACION:\n", best.to_string())

In [ ]:
# Comparacion de los dos poolings, para el README
fig, ax = plt.subplots(figsize=(7, 4))
for pooling, mk in [("mean", "o-"), ("cls", "s--")]:
    sub = search[(search["pooling"] == pooling) &
                 (search["class_weight"] == best["class_weight"])].sort_values("C")
    ax.semilogx(sub["C"], sub["macro_f1"], mk, label=f"{pooling} pooling")
ax.set_xlabel("C"); ax.set_ylabel("macro-F1 (validación)")
ax.set_title(f"Mean pooling vs CLS — {MODEL_NAME} congelado")
ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_FIGS}/fig_pooling_comparison.png", dpi=150)
plt.show()

d = search.groupby("pooling")["macro_f1"].max()
print(f"mejor mean: {d.get('mean', float('nan')):.4f} | mejor cls: {d.get('cls', float('nan')):.4f}")

## 6. Modelo final y predicciones

Se reajusta la configuración ganadora **solo con entrenamiento** (no se reentrena añadiendo
validación: eso lo decide el grupo y aquí se mantiene el protocolo común) y se guardan las
predicciones de validación y test.

Las de test **no se miran**. Se guardan para que Julia evalúe todos los modelos a la vez.

In [ ]:
pooling = best["pooling"]
cw = None if best["class_weight"] == "None" else best["class_weight"]

final = make_pipeline(
    StandardScaler(),
    LinearSVC(C=float(best["C"]), class_weight=cw, random_state=SEED, max_iter=5000),
).fit(emb[("train", pooling)], y_tr)

for nm, part, X, y in [("val", va, emb[("val", pooling)], y_va),
                       ("test", te, emb[("test", pooling)], y_te)]:
    pred   = final.predict(X)
    scores = final.decision_function(X)
    pd.DataFrame({"projectID": part["projectID"].values,
                  "y_true": y, "y_pred": pred}
                 ).to_csv(f"{OUT_PREDS}/bert_frozen_l2_{nm}.csv", index=False)
    np.save(f"{OUT_PREDS}/bert_frozen_l2_{nm}_scores.npy", scores.astype(np.float32))
    print(f"{nm}: guardadas {len(pred)} predicciones, scores {scores.shape}")

print("\n(las metricas de test NO se imprimen aqui a proposito)")

In [ ]:
# Metricas en validacion y F1 por clase (figura para el README)
from sklearn.metrics import classification_report

pred_va = final.predict(emb[("val", pooling)])
macro_f1 = f1_score(y_va, pred_va, average="macro")
micro_f1 = f1_score(y_va, pred_va, average="micro")
wtd_f1   = f1_score(y_va, pred_va, average="weighted")
acc      = accuracy_score(y_va, pred_va)
print(f"VALIDACION  macro-F1 {macro_f1:.4f} | micro-F1 {micro_f1:.4f} | "
      f"weighted-F1 {wtd_f1:.4f} | accuracy {acc:.4f}")

rep = classification_report(y_va, pred_va, output_dict=True, zero_division=0)
per_class = {int(k): v for k, v in rep.items() if k.lstrip("-").isdigit()}
order = sorted(per_class, key=lambda k: per_class[k]["support"], reverse=True)

plt.figure(figsize=(10, 5))
plt.bar(range(len(order)), [per_class[k]["f1-score"] for k in order], color="steelblue")
plt.axhline(macro_f1, color="crimson", ls="--", label=f"macro-F1 = {macro_f1:.3f}")
plt.xticks(range(len(order)),
           [NAME.get((LEVEL, str(k)), str(k))[:22] for k in order],
           rotation=90, fontsize=7)
plt.ylabel("F1"); plt.title("F1 por clase en validación (clases ordenadas por frecuencia)")
plt.legend(); plt.tight_layout()
plt.savefig(f"{OUT_FIGS}/fig_per_class_f1_bert_frozen.png", dpi=150)
plt.show()

## 7. JSON de resultados

Mismo formato que `results/tfidf_svm_l2_best.json`, para que la evaluación común lo lea
igual.

In [ ]:
import json

result = {
    "model": "bert_frozen_l2",
    "best": {
        "clf": {"C": float(best["C"]), "class_weight": best["class_weight"]},
        "emb": {
            "model_name": MODEL_NAME,
            "pooling": pooling,
            "max_length": MAX_LENGTH,
            "truncated_docs": n_over,
            "truncated_pct": round(pct_over, 4),
            "frozen": True,
        },
    },
    "val": {
        "dim": int(emb[("train", pooling)].shape[1]),
        "macro_f1": round(float(macro_f1), 6),
        "micro_f1": round(float(micro_f1), 6),
        "weighted_f1": round(float(wtd_f1), 6),
        "accuracy": round(float(acc), 6),
    },
    "n_train": int(len(y_tr)),
    "n_val": int(len(y_va)),
    "n_test": int(len(y_te)),
    "seed": SEED,
    "classes": sorted(int(c) for c in np.unique(y_tr)),
}

with open(f"{OUT_RES}/bert_frozen_l2_best.json", "w") as f:
    json.dump(result, f, indent=2)
print(json.dumps(result, indent=2))

In [ ]:
# Empaquetar lo que va al repositorio (sin embeddings ni modelos)
!cd /content/out && zip -r /content/entrega_frozen.zip results figures -x "*.pyc"
from google.colab import files
files.download("/content/entrega_frozen.zip")

## 8. Notas para el README y la defensa

Rellena esto con los números que acaban de salir:

**Elección del modelo.** `bert-base-uncased` / `roberta-base`, porque ______.

**Pooling.** Se eligió ______ por dar mejor macro-F1 en validación (___ frente a ___). La
media ponderada por la máscara de atención usa toda la secuencia, mientras que el `[CLS]`
de un modelo congelado codifica la tarea de preentrenamiento, no la nuestra; por eso suele
quedar por debajo si no hay fine-tuning.

**Textos largos.** ___ documentos (___ %) superan los 512 tokens y se truncan. Al ser una
fracción mínima, el efecto sobre las métricas es despreciable.

**Congelado frente a fine-tuning.** Congelado, el Transformer es un extractor de
características de propósito general: sus vectores no saben nada de las 39 categorías
EuroSciVoc, y toda la adaptación recae en el clasificador lineal. Con fine-tuning se
actualizan los pesos del propio Transformer con nuestras etiquetas, de modo que las
representaciones internas se reorganizan para separar precisamente esas clases. Cuesta
mucho más cómputo y tiende a dar bastante más macro-F1 (lo veremos en el notebook 04).

**Decisiones a anotar en `docs/decisiones.md`:** modelo, pooling, `max_length`, criterio de
truncado, rejilla de `C` y `class_weight`, semilla.